# Concatenación de bases de rasgos: `Rasgos_unificado` + `LeavesNapo`

**Pasos**

1. Cargar las dos bases.
2. Renombrar: el ID antiguo se llama **`old_ID`** y el ID nuevo **`new_ID`** en ambas bases.
3. **Concatenar** las dos bases.
4. **Si existe `old_ID`, ¿cuál es su `new_ID`?** Se busca en las filas que ya tienen los dos IDs (y en un archivo de equivalencias opcional) y se completa.
5. Crear un identificador único del árbol (`tree_uid`) y copiar la información fija del árbol (sitio, parcela, taxonomía…) a todos los años.
6. Control de calidad y exportación.

> Solo hay que modificar la celda de **Configuración**.

## 1. Configuración

In [30]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# --- Archivos de entrada -------------------------------------------------------
RUTA_RASGOS = Path("Rasgos_unificado_todas_las_bases.xlsx")
HOJA_RASGOS = 0
RUTA_NAPO   = Path("LeavesNapo_past.xlsx")
HOJA_NAPO   = "leaf samples Napo"

# Archivo de equivalencias OPCIONAL (columnas: Site, Plot, old_ID, new_ID).
# Si no existe, solo se usan las filas de la base que ya tienen ambos IDs.
RUTA_EQUIVALENCIAS = Path("equivalencias_old_new_ID.xlsx")

# --- Archivos de salida ---------------------------------------------------------
RUTA_SALIDA    = Path("base_concatenada.xlsx")
RUTA_PLANTILLA = Path("equivalencias_old_new_ID_PLANTILLA.xlsx")

# --- Cómo se busca el new_ID de un old_ID ----------------------------------------
# ["old_ID"]                  -> si los old_ID no se repiten en todo el proyecto
# ["Site", "old_ID"]          -> si se repiten entre sitios
# ["Site", "Plot", "old_ID"]  -> si se repiten entre parcelas (ojo: la parcela debe
#                                tener el mismo número en ambas bases)
LLAVE_OLD = ["Site", "Plot", "old_ID"]

# --- Información propia del ÁRBOL (igual en todos los años) -----------------------
COLUMNAS_ARBOL = ["Site", "PlotID", "Plot", "Subplot", "old_ID", "new_ID",
                  "family", "genus", "species", "altitude_m", "elevation_level"]

# Valores de taxonomía que cuentan como "sin dato"
VALORES_SIN_DATO = {"", "na", "nan", "none", "indet", "indet.", "sp", "sp.", "?"}

# Si un árbol tiene valores DISTINTOS entre años (p. ej. otra especie):
#   "mas_reciente" -> usar el valor del año más reciente
#   "no_tocar"     -> no cambiar nada, solo reportarlo
RESOLVER_CONFLICTOS = "mas_reciente"

## 2. Cargar las bases

In [31]:
rasgos = pd.read_excel(RUTA_RASGOS, sheet_name=HOJA_RASGOS)
napo   = pd.read_excel(RUTA_NAPO,   sheet_name=HOJA_NAPO)

print("Rasgos unificado:", rasgos.shape)
print("LeavesNapo      :", napo.shape)
display(rasgos.head())
display(napo.head())

Rasgos unificado: (313, 36)
LeavesNapo      : (416, 17)


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,Galeras,Galeras,GAL_24,24,A,NaN,8633.0,Clusiaceae,Garcinia,madruno,2025-01-08,1450.0,20.0,NaN,17.47,1568.870,89.803663,NaN,0.218000,0.684366,0.684366,93.023256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal)
1,Galeras,Galeras,GAL_24,24,B,NaN,8639.0,Lacistemataceae,Lozania,klugii,2025-01-08,1450.0,20.0,NaN,10.22,1594.462,156.013894,NaN,0.089000,0.528159,0.528159,140.476190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal)
2,Galeras,Galeras,GAL_24,24,B,NaN,8640.0,indet,NaN,NaN,2025-01-08,1450.0,20.0,NaN,9.03,645.064,71.435659,NaN,0.151111,0.396119,0.396119,197.619048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mas 10 Nutrientes,NaN,03__Rasgos_Galeras.xlsx (principal)
3,Galeras,Galeras,GAL_24,24,B,NaN,8641.0,Moraceae,Clarisia,biflora,2025-01-08,1450.0,20.0,NaN,11.94,1402.936,117.498827,NaN,0.176667,0.697666,0.697666,81.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal)
4,Galeras,Galeras,GAL_24,24,B,NaN,8642.0,Lecythidaceae,Eschweilera,caudiculata,2025-01-08,1450.0,20.0,NaN,5.06,541.494,107.014625,NaN,0.175556,0.754022,0.754022,78.947368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal)


,site,elevation level,plot ID,tree ID,year,family,genus / genus&species,species,etc,Leaf area [cm²],SLA [cm²/g],C [mg/g],N [mg/g],P [mg/g],K [mg/g],Ca [mg/g],Mg[mg/g]
0,Yanayacu,2000,1,1,2006,Cunoniaceae,Weinmannia macrophylla,NaN,NaN,24.027940,69.244784,464.49261,14.44581,0.641720,6.433433,18.630952,4.712302
1,Yanayacu,2000,1,2,2006,Staphyleaceae,Turpinia occidentalis,NaN,NaN,395.794615,100.532033,430.59620,17.44296,0.915209,14.661017,27.726820,4.497119
2,Yanayacu,2000,1,4,2006,Staphyleaceae,Turpinia occidentalis,NaN,NaN,178.433555,76.188538,426.20037,16.61254,1.440845,11.240669,35.359141,5.032338
3,Yanayacu,2000,1,7,2006,Euphorbiaceae,Alchornea cf latifolia,NaN,NaN,357.116120,100.866013,462.98466,28.37624,1.527051,17.592814,11.706587,2.696317
4,Yanayacu,2000,1,8,2006,Fabaceae,Erythrina edulis,NaN,NaN,95.198660,85.341694,464.41269,22.25334,1.048156,8.080020,14.191808,2.167213


## 3. Renombrar columnas → `old_ID` y `new_ID`

- Base unificada: `treeID` → `old_ID`, `new_tree_ID_2025` → `new_ID`.
- LeavesNapo: `tree ID` → `old_ID` y el resto de columnas al nombre que usa la base unificada.
  Revisa `RENOMBRAR_NAPO` si tu base original tiene más columnas; las que falten se conservan con un nombre limpio.

In [32]:
RENOMBRAR_RASGOS = {
    "treeID": "old_ID",
    "new_tree_ID_2025": "new_ID",
}

RENOMBRAR_NAPO = {
    "site": "Site",
    "elevation level": "elevation_level",
    "plot ID": "Plot",
    "tree ID": "old_ID",
    "year": "year",
    "family": "family",
    "genus / genus&species": "genus",
    "species": "species",
    "etc": "etc",
    "Leaf area [cm²]": "leaf_area_cm2",
    "SLA [cm²/g]": "SLA_cm2_g",
    "C [mg/g]": "C_mg_g",
    "N [mg/g]": "N_mg_g",
    "P [mg/g]": "P_mg_g",
    "K [mg/g]": "K_mg_g",
    "Ca [mg/g]": "Ca_mg_g",
    "Mg[mg/g]": "Mg_mg_g",
}

def limpiar_nombre(c):
    return (str(c).strip().replace(" ", "_").replace("[", "").replace("]", "")
            .replace("/", "_").replace("²", "2"))

rasgos_h = rasgos.rename(columns=RENOMBRAR_RASGOS).copy()
napo_h = napo.rename(columns=lambda c: RENOMBRAR_NAPO.get(str(c).strip(), limpiar_nombre(c))).copy()

no_mapeadas = [c for c in napo.columns if str(c).strip() not in RENOMBRAR_NAPO]
if no_mapeadas:
    print("⚠️  Columnas de Napo sin mapeo explícito (se conservan con nombre limpio):", no_mapeadas)

# Columnas que identifican de dónde viene cada fila
rasgos_h["year"] = pd.to_datetime(rasgos_h["sampling_date"], errors="coerce").dt.year
napo_h["dataset_source"] = "LeavesNapo"
napo_h["source_file"] = RUTA_NAPO.name

## 4. Concatenar las dos bases

In [33]:
base = pd.concat([rasgos_h, napo_h], ignore_index=True, sort=False)
print("Base concatenada:", base.shape)
print("Columnas que solo existen en LeavesNapo:",
      [c for c in napo_h.columns if c not in rasgos_h.columns])
base.head(5)

Base concatenada: (729, 39)
Columnas que solo existen en LeavesNapo: ['elevation_level', 'etc']


,dataset_source,Site,PlotID,Plot,Subplot,old_ID,new_ID,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file,year,elevation_level,etc
0,Galeras,Galeras,GAL_24,24,A,NaN,8633.0,Clusiaceae,Garcinia,madruno,2025-01-08,1450.0,20.0,NaN,17.47,1568.870,89.803663,NaN,0.218000,0.684366,0.684366,93.023256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal),2025.0,NaN,NaN
1,Galeras,Galeras,GAL_24,24,B,NaN,8639.0,Lacistemataceae,Lozania,klugii,2025-01-08,1450.0,20.0,NaN,10.22,1594.462,156.013894,NaN,0.089000,0.528159,0.528159,140.476190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal),2025.0,NaN,NaN
2,Galeras,Galeras,GAL_24,24,B,NaN,8640.0,indet,NaN,NaN,2025-01-08,1450.0,20.0,NaN,9.03,645.064,71.435659,NaN,0.151111,0.396119,0.396119,197.619048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mas 10 Nutrientes,NaN,03__Rasgos_Galeras.xlsx (principal),2025.0,NaN,NaN
3,Galeras,Galeras,GAL_24,24,B,NaN,8641.0,Moraceae,Clarisia,biflora,2025-01-08,1450.0,20.0,NaN,11.94,1402.936,117.498827,NaN,0.176667,0.697666,0.697666,81.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal),2025.0,NaN,NaN
4,Galeras,Galeras,GAL_24,24,B,NaN,8642.0,Lecythidaceae,Eschweilera,caudiculata,2025-01-08,1450.0,20.0,NaN,5.06,541.494,107.014625,NaN,0.175556,0.754022,0.754022,78.947368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03__Rasgos_Galeras.xlsx (principal),2025.0,NaN,NaN


## 5. Limpiar tipos y formatos

- IDs, parcela y año como enteros (para que `1812` y `1812.0` sean el mismo ID).
- `Site` sin espacios y con mayúscula inicial.
- Taxonomía: “indet”, “sp.”… pasan a vacío; si `genus` trae “Género especie” junto, se separa.
- `PlotID` se completa con el prefijo del sitio (p. ej. `GAL_60`).

In [34]:
def a_entero(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

for col in ["old_ID", "new_ID", "Plot", "year"]:
    base[col] = a_entero(base[col])

base["Site"] = base["Site"].astype("string").str.strip().str.title()

for col in ["family", "genus", "species"]:
    t = base[col].astype("string").str.strip()
    base[col] = t.mask(t.str.lower().isin(VALORES_SIN_DATO))

junto = base["genus"].str.contains(r"\s", na=False) & base["species"].isna()
if junto.any():
    partes = base.loc[junto, "genus"].str.split(n=1, expand=True)
    base.loc[junto, "genus"], base.loc[junto, "species"] = partes[0], partes[1]
    print(f"Separadas {junto.sum()} filas con 'Género especie' en una sola columna.")

# Prefijo de PlotID por sitio, aprendido de las filas que ya lo tienen (GAL_24 -> GAL)
prefijos = (base.dropna(subset=["PlotID"])
            .assign(pref=lambda d: d["PlotID"].astype(str).str.split("_").str[0])
            .groupby("Site")["pref"].agg(lambda x: x.mode().iat[0]))
falta = base["PlotID"].isna() & base["Plot"].notna()
base.loc[falta, "PlotID"] = base.loc[falta, "Site"].map(prefijos) + "_" + base.loc[falta, "Plot"].astype("string")
print("Prefijos de PlotID por sitio:", prefijos.to_dict())

Separadas 204 filas con 'Género especie' en una sola columna.
Prefijos de PlotID por sitio: {'Bon': 'BoN', 'Canu': 'CaNu', 'Galeras': 'GAL', 'Guacamayos': 'GUA08', 'Oyacachi': 'OYC-81', 'Selva Viva': 'SEV', 'Sfnu': 'SFNu', 'Sumaco': 'SUM'}


In [ ]:
df = base
# 2. Seleccionar las filas de 2025 con ambos IDs disponibles

ref_2025 = df.loc[
    df["year"].isin([2023, 2024, 2025]) &
    df["old_ID"].notna() &
    df["new_ID"].notna(),
    ["old_ID", "new_ID"]
].copy()

# 3. Comprobar que cada old_ID corresponde a un único new_ID
check = ref_2025.groupby("old_ID")["new_ID"].nunique()

conflictos = check[check > 1]

if not conflictos.empty:
    raise ValueError(
        f"Hay old_ID asociados a varios new_ID en 2025: "
        f"{conflictos.index.tolist()}"
    )

# 4. Crear el diccionario old_ID -> new_ID
id_mapping = (
    ref_2025.drop_duplicates(subset="old_ID")
    .set_index("old_ID")["new_ID"]
)

# 5. Asignar el new_ID correspondiente a toda la base
# Solo rellenar los new_ID que están vacíos
df["new_ID"] = df["new_ID"].fillna(
    df["old_ID"].map(id_mapping)
)


In [37]:
# List of sites to exclude
exclude_sites = ["Yanayacu", "Hakuna Matata","Guango"]

# Filtering out these ages using ~isin()
df_filtered = df[~df['Site'].isin(exclude_sites)]


# 6. Guardar la base actualizada
df_filtered.to_excel("BASE_actualizada.xlsx", index=False)